In [1]:
!pip install ultralytics roboflow opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


In [2]:
import os
HOME = os.getcwd()

In [3]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.5/112.6 GB disk)


In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="bgZz4XsPKrewZX7Qhbab")
project = rf.workspace("ahmeds-workspace-slwao").project("bike_fit")
version = project.version(4)
dataset = version.download("yolov8")




loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bike_fit-3 in yolov8:: 100%|██████████| 117/117 [00:00<00:00, 6246.77it/s]


In [8]:
from ultralytics import YOLO

model  = YOLO('yolo11n-pose.pt')

results = model.train(data="bike_fit-3/data.yaml", epochs=300, imgsz=640  )

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bike_fit-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

In [83]:
res = model('/content/frame_760.jpg' , save = True)


image 1/1 /content/frame_760.jpg: 416x640 1 points, 9.4ms
Speed: 2.3ms preprocess, 9.4ms inference, 1.6ms postprocess per image at shape (1, 3, 416, 640)
Results saved to /content/runs/pose/predict-2


In [84]:
import numpy as np
import math

def calculate_angle(A, B, C):
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    BA = A - B
    BC = C - B

    dot_product = np.dot(BA, BC)

    magnitude_BA = np.linalg.norm(BA)
    magnitude_BC = np.linalg.norm(BC)

    cos_angle = dot_product / (magnitude_BA * magnitude_BC)

    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    angle = np.arccos(cos_angle)

    angle = np.degrees(angle)

    angle1 = math.degrees(
    math.atan2(A[1] - B[1], A[0] - B[0])
    )

    angle2 = math.degrees(
    math.atan2(C[1] - B[1], C[0] - B[0])
    )
    radius = 40

    start_angle = int(angle1)
    end_angle = int(angle2)

    return angle , start_angle , end_angle , radius

In [85]:
def feedback(angles) :
  notes = {}
  elpow_angle , shoulder_ange  , hip_angle , knee_angle = angles
  if int(elpow_angle) in range(150 , 170) :
    notes['Elbow angle'] = 'within range'
  elif int(elpow_angle) < 150 :
    notes['Elbow angle'] = 'Try longer Stem'
  elif int(elpow_angle) > 170 :
    notes['Elbow angle'] = 'Try shorter Stem'

  if int(shoulder_ange) in range(70 , 95) :
    notes['Shoulder angle'] = 'within range'
  elif int(shoulder_ange) < 70 :
    notes['Shoulder angle'] = 'Try moving the seat further back'
  elif int(shoulder_ange) > 95 :
    notes['Shoulder angle'] = 'Try moving the seat forword'

  if int(hip_angle) in range(60 , 125) :
    notes['Hip angle'] = 'within range'
  elif int(shoulder_ange) < 60 :
    notes['Hip angle'] = 'Try elevate the seat'
  elif int(shoulder_ange) > 125 :
    notes['Hip angle'] = 'Try lowering the seat '

  if int(knee_angle) in range(60 , 150) :
    notes['Knee angle'] = 'within range'
  elif int(shoulder_ange) < 60 :
    notes['Knee angle'] = 'Try elevate the seat'
  elif int(shoulder_ange) > 125 :
    notes['Knee angle'] = 'Try lowering the seat '

  return notes

In [86]:

import cv2
img = cv2.imread("/content/frame_760.jpg")


lines = [
    (0, 1), (1, 2),
    (2, 3), (3, 4),
    (4, 5),
]
angles = [
    (0 , 1, 2) ,
    (1,2,3) ,
    (2,3,4) ,
    (3,4,5)
]

for result in res:

    # Get keypoints
    keypoints = result.keypoints.xy.cpu().numpy()

    # Loop through detected persons/objects
    for person in keypoints:

        # Draw keypoints
        for i, (x, y) in enumerate(person):

            x = int(x)
            y = int(y)

            cv2.circle(img, (x, y), 5, (0, 255, 0), -1)


        for p1, p2 in lines:

            x1, y1 = person[p1]
            x2, y2 = person[p2]

            cv2.line(
                img,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (0, 0, 255),
                2
            )
        Angles = []
        for A1 , A2 ,A3 in angles :
            angle , start_angle , end_angle , radius = calculate_angle(person[A1] , person[A2] , person[A3])
            Angles.append(angle)
            cv2.ellipse(
                img,
                (int(person[A2][0]) , int(person[A2][1])),                      # center
                (radius, radius),       # axes
                0,                      # rotation
                start_angle,
                end_angle,
                (255, 0, 0),          # yellow arc
                2
              )
            cv2.putText(
              img,
              f"{int(angle)} o",
              (int(person[A2][0]) +20 , int(person[A2][1])-20),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.7,
              (0, 0, 0),
              2
            )
        notes = feedback(Angles)
        for  i , (A , n) in enumerate(notes.items()) :
          cv2.putText(
              img,
              f"{A}:{n}",
              ((30) , (i*20)),
              cv2.FONT_HERSHEY_SIMPLEX,
              0.5,
              (0, 0, 255),
              2
            )
# Save result
cv2.imwrite("output.jpg", img)

# Display
# cv2.imshow("Pose", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

error: OpenCV(4.10.0) /io/opencv/modules/highgui/src/window.cpp:1367: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvWaitKey'


In [38]:
import numpy as np
import math

def angle_between_lines(A, B, C, D):

    # Convert to vectors
    v1 = np.array(B) - np.array(A)
    v2 = np.array(D) - np.array(C)

    # Dot product
    dot_product = np.dot(v1, v2)

    # Magnitudes
    mag1 = np.linalg.norm(v1)
    mag2 = np.linalg.norm(v2)

    # Cosine of angle
    cos_theta = dot_product / (mag1 * mag2)

    # Avoid numerical errors
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    # Angle in radians
    angle_rad = math.acos(cos_theta)

    # Convert to degrees
    angle_deg = math.degrees(angle_rad)

    return angle_deg

In [63]:
notes = {'w' : '2' , 'X' : '1'}

In [66]:
for key , value in notes.items() :
  print(key)
  print(value)

w
2
X
1
